<a href="https://colab.research.google.com/github/ElizabethFrankWebb/USRI-2026/blob/main/Individual_Base_Model_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import numpy as np
import matplotlib.pyplot
import random

In [21]:
#parameters

intial_population = 100
loci = 5
selection = 10
optimal_trait_value = 0
mutation_rate = 10**-2
generation = 10
genotypes = np.zeros((intial_population, loci))

In [22]:
# Calculate trait value for all individuals

def calculate_trait_z(individual_genotype):
  trait_value = np.sum(individual_genotype)
  return trait_value
trait_values = np.array([calculate_trait_z(individual) for individual in genotypes])

print("Trait values for all individuals:")
print(trait_values)


Trait values for all individuals:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0.]


In [26]:
def calculate_fitness(trait_values, optimal_trait_value, selection):
  trait_values = np.asarray(trait_values)
  fitness = np.exp(-(trait_values - optimal_trait_value)**2 / (2 * selection))
  return fitness # Added return statement here

population_fitness = calculate_fitness(trait_values, optimal_trait_value, selection)

print(population_fitness)

[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1.]


In [27]:
#next generation with asexual mating using poisson distrbution for number of offsring

base_rate = 2.0
lambdas = base_rate * population_fitness

offspring_counts = np.random.poisson(lambdas)

print(offspring_counts)

[0 2 2 5 3 1 5 2 1 0 3 3 0 1 2 3 1 2 3 2 0 2 3 1 0 0 0 2 3 2 0 0 2 0 2 3 3
 2 2 1 5 0 3 1 5 2 1 3 2 3 3 3 1 3 1 2 2 4 4 2 1 1 1 2 1 2 2 3 4 0 1 3 3 2
 2 3 0 1 0 2 3 2 6 2 0 1 1 2 1 1 0 2 0 0 0 2 3 3 2 3]


In [36]:
# introducing mutation

def mutate(individual, mutation_rate):
  mutated_individual = individual.copy()
  mutated_loci_info = [] # Store (locus_index, original_value, new_value)
  for locus in range(len(individual)):
    if np.random.rand() < mutation_rate:
      original_value = individual[locus] # Get original value before mutation
      new_value = np.random.normal(0,1) # Assign new value from normal distribution (mean=0, std=1)
      mutated_individual[locus] = new_value
      mutated_loci_info.append((locus, original_value, new_value))
  return mutated_individual, mutated_loci_info # Return both the mutated individual and info about mutations

In [37]:
# Generate the next generation with mutations
new_gen_genotypes_list = []
offspring_mutation_details = []
offspring_counter = 0

# Ensure intial_population is updated to reflect the current size of genotypes array
current_population_size = len(genotypes)

for parent_idx in range(current_population_size):
    parent_genotype = genotypes[parent_idx]
    num_offspring = offspring_counts[parent_idx]

    for _ in range(num_offspring):
        mutated_individual, mutated_loci_info = mutate(parent_genotype, mutation_rate)
        new_gen_genotypes_list.append(mutated_individual)

        if mutated_loci_info:
            offspring_mutation_details.append({
                'offspring_idx': offspring_counter,
                'parent_idx': parent_idx,
                'mutations': mutated_loci_info
            })
        offspring_counter += 1

# Update the genotypes for the new generation
if new_gen_genotypes_list:
    genotypes = np.array(new_gen_genotypes_list)
    intial_population = len(genotypes) # Update population size for the next generation
else:
    genotypes = np.empty((0, loci)) # Handle case with no offspring
    intial_population = 0

print("--- New Generation Genotypes with Mutations ---")
if intial_population > 0:
    for i, individual_genotype in enumerate(genotypes):
        print(f"Offspring {i}: Genotype = {individual_genotype}")
else:
    print("No offspring were generated in this generation.")

print("\n--- Mutation Details ---")
if offspring_mutation_details:
    for detail in offspring_mutation_details:
        print(f"Offspring {detail['offspring_idx']} (from Parent {detail['parent_idx']}) mutated at:")
        for locus_idx, original_val, new_val in detail['mutations']:
            print(f"  Locus {locus_idx}: Changed from {original_val:.4f} to {new_val:.4f}")
else:
    print("No mutations occurred in this generation.")

# Recalculate trait values for the new generation
if intial_population > 0:
    trait_values = np.array([calculate_trait_z(individual) for individual in genotypes])
    print(f"\nRecalculated Trait values for new generation ({intial_population} individuals):")
    print(trait_values)

    # Recalculate fitness for the new generation
    population_fitness = calculate_fitness(trait_values, optimal_trait_value, selection)
    print(f"\nRecalculated Fitness for new generation ({intial_population} individuals):")
    print(population_fitness)
else:
    print("\nCannot recalculate trait values or fitness as no offspring were generated.")

--- New Generation Genotypes with Mutations ---
Offspring 0: Genotype = [0. 0. 0. 0. 0.]
Offspring 1: Genotype = [0.         1.31632794 0.         0.         0.        ]
Offspring 2: Genotype = [0. 0. 0. 0. 0.]
Offspring 3: Genotype = [0. 0. 0. 0. 0.]
Offspring 4: Genotype = [0. 0. 0. 0. 0.]
Offspring 5: Genotype = [0. 0. 0. 0. 0.]
Offspring 6: Genotype = [0. 0. 0. 0. 0.]
Offspring 7: Genotype = [0.         0.         0.52826696 0.         0.        ]
Offspring 8: Genotype = [0. 0. 0. 0. 0.]
Offspring 9: Genotype = [0. 0. 0. 0. 0.]
Offspring 10: Genotype = [0. 0. 0. 0. 0.]
Offspring 11: Genotype = [0. 0. 0. 0. 0.]
Offspring 12: Genotype = [0. 0. 0. 0. 0.]
Offspring 13: Genotype = [0. 0. 0. 0. 0.]
Offspring 14: Genotype = [0. 0. 0. 0. 0.]
Offspring 15: Genotype = [0. 0. 0. 0. 0.]
Offspring 16: Genotype = [0.         0.         0.         1.08581733 0.        ]
Offspring 17: Genotype = [0. 0. 0. 0. 0.]
Offspring 18: Genotype = [0. 0. 0. 0. 0.]
Offspring 19: Genotype = [0. 0. 0. 0. 0.]
Of

### Simulation Loop

In [39]:
for g in range(generation):
    print(f"\n{'='*30} GENERATION {g+1} {'='*30}")

    # Recalculate lambdas for offspring counts based on current population fitness
    lambdas = base_rate * population_fitness
    offspring_counts = np.random.poisson(lambdas)

    # Generate the next generation with mutations
    new_gen_genotypes_list = []
    offspring_mutation_details = []
    offspring_counter = 0

    # Ensure intial_population is updated to reflect the current size of genotypes array
    current_population_size = len(genotypes)

    for parent_idx in range(current_population_size):
        parent_genotype = genotypes[parent_idx]
        num_offspring = offspring_counts[parent_idx]

        for _ in range(num_offspring):
            mutated_individual, mutated_loci_info = mutate(parent_genotype, mutation_rate)
            new_gen_genotypes_list.append(mutated_individual)

            if mutated_loci_info:
                offspring_mutation_details.append({
                    'offspring_idx': offspring_counter,
                    'parent_idx': parent_idx,
                    'mutations': mutated_loci_info
                })
            offspring_counter += 1

    # Update the genotypes for the new generation
    if new_gen_genotypes_list:
        genotypes = np.array(new_gen_genotypes_list)
        intial_population = len(genotypes) # Update population size for the next generation
    else:
        genotypes = np.empty((0, loci)) # Handle case with no offspring
        intial_population = 0

    print("--- New Generation Genotypes with Mutations ---")
    if intial_population > 0:
        for i, individual_genotype in enumerate(genotypes):
            print(f"Offspring {i}: Genotype = {individual_genotype}")
    else:
        print("No offspring were generated in this generation.")

    print("\n--- Mutation Details ---")
    if offspring_mutation_details:
        for detail in offspring_mutation_details:
            print(f"Offspring {detail['offspring_idx']} (from Parent {detail['parent_idx']}) mutated at:")
            for locus_idx, original_val, new_val in detail['mutations']:
                print(f"  Locus {locus_idx}: Changed from {original_val:.4f} to {new_val:.4f}")
    else:
        print("No mutations occurred in this generation.")

    # Recalculate trait values for the new generation
    if intial_population > 0:
        trait_values = np.array([calculate_trait_z(individual) for individual in genotypes])
        print(f"\nRecalculated Trait values for new generation ({intial_population} individuals):")
        print(trait_values)

        # Recalculate fitness for the new generation
        population_fitness = calculate_fitness(trait_values, optimal_trait_value, selection)
        print(f"\nRecalculated Fitness for new generation ({intial_population} individuals):")
        print(population_fitness)
    else:
        print("\nCannot recalculate trait values or fitness as no offspring were generated.")

Streaming output truncated to the last 5000 lines.
  Locus 2: Changed from 0.0000 to -1.0184
Offspring 116301 (from Parent 59123) mutated at:
  Locus 3: Changed from -0.0655 to -1.4657
Offspring 116366 (from Parent 59162) mutated at:
  Locus 4: Changed from 0.0000 to 1.4335
Offspring 116389 (from Parent 59175) mutated at:
  Locus 1: Changed from 0.0000 to -1.1497
Offspring 116398 (from Parent 59178) mutated at:
  Locus 1: Changed from 0.0000 to 0.0540
Offspring 116400 (from Parent 59179) mutated at:
  Locus 3: Changed from 0.0000 to -0.7206
Offspring 116434 (from Parent 59193) mutated at:
  Locus 4: Changed from 0.0000 to 1.5324
Offspring 116473 (from Parent 59219) mutated at:
  Locus 0: Changed from 0.0000 to -0.7524
Offspring 116486 (from Parent 59226) mutated at:
  Locus 3: Changed from 0.0000 to 0.6827
Offspring 116496 (from Parent 59230) mutated at:
  Locus 1: Changed from 0.0000 to 0.3164
Offspring 116506 (from Parent 59236) mutated at:
  Locus 4: Changed from 0.0000 to -0.2263
O